# A Globus endpoint on FABRIC

Builds a **Globus Connect Server 5.4** data transfer node (DTN) on a FABRIC VM and publishes
one directory on it as a Globus **mapped collection**. Two ways to make the VM reachable from
the internet, chosen with one setting (`profile` in `dtn_config.py`):

| profile | public address from | you need |
|---|---|---|
| `ext` | **FABnetv4Ext** - FABRIC's own routable IPv4 | nothing outside FABRIC |
| `fp` | a partner's **L3 facility port** (e.g. `NCSA-L3-NCSA`) | a VLAN, prefix and gateway from the partner |

```
   Globus user / another endpoint
              |  443 (HTTPS, control)   50000-51000 (GridFTP data)
              v
   +----------- globus-dtn (Rocky 9) ---------------------------+
   |  eth0   FABRIC management   - untouched, keeps SSH working |
   |  ethX   public uplink       - FABnetv4Ext  or  facility port|
   |         own routing table + source rule; firewalled to     |
   |         443 and 50000-51000 only                           |
   |  /data/<account>            - the mapped collection        |
   +------------------------------------------------------------+
```

**Before you start**

1. FABRIC: a project and a working FABlib config (`fabric_rc`, token, bastion key) -
   the same setup as any FABRIC notebook.
2. Globus: a **service credential for Globus Connect Server** - at app.globus.org >
   Settings > Developers, create a project, then *Register a service account / application
   credential for Globus Connect Server*. Copy `globus_creds.env.example` to `globus_creds.env`
   and fill in its id and secret. The endpoint is owned by that credential, not by a person.
3. Edit `dtn_config.py`: the profile, your organization and contact, `allowed_domains`, and
   `users` (which Globus identities may use the collection, and as which local account).
4. For `fp`: the partner's address assignment, or run the probe step first.

Every cell re-reads the slice, so any cell can be re-run on its own.

## Configuration and FABlib

In [ ]:
import importlib, json, os, shlex
from ipaddress import IPv4Network
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

import dtn_config as cfg
importlib.reload(cfg)
cfg.validate()

fablib = fablib_manager()
fablib.show_config();

print(f"profile  : {cfg.profile}  ({cfg.uplink} at {cfg.site})")
print(f"slice    : {cfg.slice_name}, node {cfg.node_name}")
print(f"endpoint : '{cfg.endpoint_display_name}', owned by {cfg.owner_identity()}")
print(f"storage  : {cfg.data_base} for {len(cfg.users)} user(s), domains {cfg.allowed_domains}")

def ctx():
    """(slice, node, public device) - re-read every time, so cells are independent."""
    s = fablib.get_slice(name=cfg.slice_name)
    n = s.get_node(name=cfg.node_name)
    return s, n, n.get_interface(network_name=cfg.ext_net_name).get_device_name()

def run(node, cmd):
    """node.execute() that FAILS the cell when the remote command fails.

    FABlib's execute() returns output and never raises on a non-zero exit, so a
    failed step would otherwise look like success and the next cell would run
    on a half-built endpoint."""
    out, err = node.execute(f"{cmd}; echo __rc=$?")
    rc = int(out.strip().rsplit("__rc=", 1)[-1]) if "__rc=" in out else -1
    if rc != 0:
        raise RuntimeError(f"remote command failed (exit {rc}) - see the output above")
    return out, err

## Facility port: what FABRIC advertises (`fp` only)

FABRIC publishes each facility port's VLAN range and which VLANs are taken - nothing about
addressing. This reads it live, and resolves `vlan="auto"` to the first free VLAN.

In [ ]:
vlan = cfg.vlan
if cfg.uplink == "facility_port":
    import ast
    rows = fablib.get_facility_ports().list_facility_ports(output="list", quiet=True)
    print("L3 facility ports right now:")
    for r in rows:
        if "L3" in r["name"] or "Layer3" in r["name"]:
            print(f"  {r['name']:22s} {r['site']:6s} vlans {r['vlans']}  in use {r.get('allocated_vlans', '[]')}")
    mine = [r for r in rows if r["name"] == cfg.facility_port and r["site"] == cfg.site]
    assert mine, f"{cfg.facility_port} is not advertised at {cfg.site}"
    ranges = ast.literal_eval(mine[0]["vlans"])
    taken = {int(v) for v in ast.literal_eval(mine[0].get("allocated_vlans", "[]"))}
    every = [v for rg in ranges for v in range(int(rg.split("-")[0]), int(rg.split("-")[-1]) + 1)]
    if vlan == "auto":
        vlan = next(v for v in every if v not in taken)
        print(f"\nvlan 'auto' -> {vlan}. Tell the partner which VLAN you are on.")
    else:
        vlan = int(vlan)
        assert vlan in every, f"VLAN {vlan} is outside {ranges}"
        if vlan in taken:
            print(f"\nWARN: VLAN {vlan} is already allocated - the submit will fail unless it is yours.")
else:
    print("ext profile - no facility port involved")

## Create the slice

One Rocky 9 VM with a dedicated NIC for the public side. For `ext` that NIC joins a
FABnetv4Ext network; for `fp` it is bridged at layer 2 to the facility port on one VLAN.

In [ ]:
try:
    slice = fablib.get_slice(name=cfg.slice_name)
    print(f"slice {cfg.slice_name} exists - reusing it")
except Exception:
    slice = fablib.new_slice(name=cfg.slice_name)
    node = slice.add_node(name=cfg.node_name, site=cfg.site, cores=cfg.cores,
                          ram=cfg.ram, disk=cfg.disk, image=cfg.image)
    iface = node.add_component(model=cfg.nic_model, name="pub-nic").get_interfaces()[0]
    if cfg.uplink == "fabnetv4ext":
        slice.add_l3network(name=cfg.ext_net_name, interfaces=[iface], type="IPv4Ext")
    else:
        fp = slice.add_facility_port(name=cfg.facility_port, site=cfg.site, vlan=str(vlan),
                                     bandwidth=cfg.fp_bw, mtu=cfg.fp_mtu)
        slice.add_l2network(name=cfg.ext_net_name, interfaces=[iface, fp.get_interfaces()[0]])
    node.add_post_boot_upload_directory("node_tools", ".")
    slice.submit();

### Renew the lease right away

A new slice gets about a day. Renewing straight after `submit()` avoids losing the endpoint
over a weekend.

In [ ]:
slice = fablib.get_slice(name=cfg.slice_name)
slice.renew(days=14)
print(f"lease ends: {slice.get_lease_end()}")

## Probe the facility port (`fp` only, optional)

Before - or while - asking the partner for addresses, see what the port offers on your VLAN.
`probe_facility_port.sh` brings the interface up on a throwaway profile that asks for DHCPv4
and IPv6 autoconfiguration. Everything it learns, default routes included, goes into a side
routing table, so management is never touched; it refuses to run if that isolation cannot be
guaranteed. It pings out, asks `ifconfig.me` which address the far side sees (**a different
address means NAT, and Globus cannot be reached inbound through NAT**), and removes everything.

With `--sniff`, if nothing was assigned, it captures 20 s of ARP/RA/DHCP traffic. Seeing the
partner's router ARP for addresses in a block is a strong hint that the block is routed to your
VLAN and expects static addressing. Use only addresses the partner assigns to you.

In [ ]:
if cfg.uplink == "facility_port":
    slice, node, dev = ctx()
    node.upload_directory("node_tools", ".")
    node.execute("sudo dnf install -y -q tcpdump >/dev/null 2>&1 || true", quiet=True)
    stdout, stderr = run(node, f"sudo bash node_tools/probe_facility_port.sh --iface {dev} --wait 45 --sniff 20")
else:
    print("ext profile - nothing to probe")

## The public address

**ext:** a FABnetv4Ext address becomes routable only when asked, which takes a **second
`submit()`**. Always read it back - if your first choice is taken, FABRIC hands you the next
free one.

**fp:** the partner's assignment from `dtn_config.py`. This cell stops until at least one
family is filled in.

In [ ]:
slice, node, dev = ctx()
addr4, gw4, addr6, gw6 = cfg.addr4, cfg.gw4, cfg.addr6, cfg.gw6
if cfg.uplink == "fabnetv4ext":
    net = slice.get_network(name=cfg.ext_net_name)
    if not net.get_public_ips():
        net.make_ip_publicly_routable(ipv4=[str(net.get_available_ips()[0])])
        slice.submit()
        slice = fablib.get_slice(name=cfg.slice_name)
        net = slice.get_network(name=cfg.ext_net_name)
    addr4 = f"{net.get_public_ips()[0]}/{net.get_subnet().prefixlen}"
    gw4 = str(net.get_gateway())
else:
    cfg.validate(need_addressing=True)

public_ip = addr4.split("/")[0] if addr4 else None
public_ip6 = addr6.split("/")[0] if addr6 else None
print(f"IPv4: {addr4 or '-'} via {gw4 or '-'}")
print(f"IPv6: {addr6 or '-'} via {gw6 or '-'}")

## Configure the uplink

`setup_globus_dtn_net.sh` gives the public address its own routing table and a source rule,
so replies to anything arriving on it leave the same way, while FABRIC management keeps its
own default route. Per address family: if management has **no** default route in that family
(FABRIC management is often IPv6-only), the uplink also becomes that family's default -
otherwise outbound IPv4 (package installs, Globus API calls) would have nowhere to go.
Persisted in NetworkManager and re-asserted at boot by `globus-dtn-pbr.service`.

In [ ]:
slice, node, dev = ctx()
node.upload_directory("node_tools", ".")
args = f"--iface {dev}"
if addr4: args += f" --addr {addr4} --gw {gw4}"
if addr6: args += f" --addr6 {addr6} --gw6 {gw6}"
if cfg.fp_mtu: args += f" --mtu {cfg.fp_mtu}"
stdout, stderr = run(node, f"sudo node_tools/setup_globus_dtn_net.sh {args}")
run(node, f"sudo node_tools/host_tune.sh" + (f" --mtu-iface {dev} --mtu {cfg.fp_mtu}" if cfg.fp_mtu else ""));

In [ ]:
# Out through the public address, and the address the far side sees. A different one
# means NAT - and then no Globus client can reach this endpoint.
slice, node, dev = ctx()
for fam, ip, target in (("4", public_ip, "8.8.8.8"), ("6", public_ip6, "2001:4860:4860::8888")):
    if ip:
        node.execute(f"ping -{fam} -c3 -I {ip} {target} | tail -1; "
                     f"echo \"seen from outside as: $(curl -{fam} -s --max-time 8 --interface {ip} https://ifconfig.me)\"")

## Install Globus Connect Server

In [ ]:
slice, node, dev = ctx()
node.upload_directory("node_tools", ".")   # the VM runs these copies - push any edits
stdout, stderr = run(node, f"sudo node_tools/install_globus_gcs.sh --wan-iface {dev}")

### Let only Globus in on the public interface

The public address answers for everything listening on all addresses - sshd included. This
installs one nftables table matching **only** the public interface: replies to the VM's own
connections, the ICMP IPv6 needs, and TCP 443 + 50000-51000 are allowed, everything else
arriving there is dropped. Management is never matched, and the script refuses if your session
arrives through the interface it would filter. `--remove` undoes it.

In [ ]:
slice, node, dev = ctx()
stdout, stderr = run(node, 
    f'sudo SSH_CONNECTION="$SSH_CONNECTION" node_tools/setup_globus_wan_firewall.sh --iface {dev}')

## Register the endpoint

`endpoint setup` creates the endpoint in Globus's cloud and a `deployment-key.json` on the VM;
`node setup` attaches this VM to it at the public address. Running `endpoint setup` twice would
create a second endpoint, so the script refuses once the key exists. Globus assigns a
`*.gaccess.io` name and its certificate - no DNS work.

In [ ]:
slice, node, dev = ctx()
node.upload_directory("node_tools", ".")   # the VM runs these copies - push any edits
node.upload_file(cfg.globus_creds_file, "globus_creds.env")
node.execute(f"sudo install -d -m 0755 {cfg.state_dir} && "
             f"sudo install -m 0600 globus_creds.env {cfg.state_dir}/creds.env && "
             f"shred -u globus_creds.env && echo credentials staged")

cmd = (f"sudo node_tools/setup_globus_endpoint.sh "
       f"--creds-file {cfg.state_dir}/creds.env --state-dir {cfg.state_dir} "
       f"--display-name {shlex.quote(cfg.endpoint_display_name)} "
       f"--organization {shlex.quote(cfg.organization)} "
       f"--contact-email {shlex.quote(cfg.contact_email)} "
       f"--owner {shlex.quote(cfg.owner_identity())} --dont-set-advertised-owner "
       f"--public-ip {public_ip or public_ip6}")
if cfg.globus_project_id: cmd += f" --project-id {cfg.globus_project_id}"
if public_ip and public_ip6: cmd += f" --public-ip6 {public_ip6}"
stdout, stderr = run(node, cmd)

### Keep the endpoint's identity off the VM

`deployment-key.json` **is** the endpoint. With a copy, a replacement VM runs `node setup` and
rejoins the same endpoint - same id, domain and collections. Without it, losing the VM (FABRIC
can close a VM for maintenance or capacity) means a new endpoint and every collection
re-created under new ids. Saved beside this notebook as `deployment-key-<site>.json` (0600,
gitignored). Treat it as a secret.

In [ ]:
slice, node, dev = ctx()
key_copy = f"deployment-key-{cfg.site}.json"
node.execute(f"sudo install -m 0600 -o $(id -un) {cfg.state_dir}/deployment-key.json dk.json", quiet=True)
_old = os.umask(0o077)
try:
    node.download_file(key_copy, "dk.json")
finally:
    os.umask(_old)
    node.execute("shred -u dk.json", quiet=True)
stdout, _ = node.execute(f"sudo cat {cfg.state_dir}/endpoint-id.txt", quiet=True)
assert json.load(open(key_copy))["client_id"] == stdout.strip(), "copied key is not this endpoint's"
print(f"saved {key_copy} for endpoint {stdout.strip()}")

## Publish a directory

`setup_globus_storage.sh` creates, for each entry in `users`, a local account and a private
`<data_base>/<account>` directory; writes an identity map with **exact** matches only (an
identity not listed is refused); and creates one POSIX storage gateway plus one mapped
collection rooted at `data_base`. Transfers run as the mapped account, so normal POSIX
permissions apply. Re-run it after changing `users`.

The collection lives on the VM's own disk. For more space, add an NVMe component to the node
and mount it at `data_base` before this step.

In [ ]:
slice, node, dev = ctx()
node.upload_directory("node_tools", ".")
cmd = (f"sudo node_tools/setup_globus_storage.sh --base {shlex.quote(cfg.data_base)} "
       f"--gateway-name {shlex.quote(cfg.endpoint_display_name + ' storage')} "
       f"--collection-name {shlex.quote(cfg.endpoint_display_name)} "
       f"--creds-file {cfg.state_dir}/creds.env --state-dir {cfg.state_dir}")
cmd += "".join(f" --domain {shlex.quote(d)}" for d in cfg.allowed_domains)
cmd += "".join(f" --map {shlex.quote(i + '=' + a)}" for i, a in cfg.users.items())
if cfg.globus_admin: cmd += f" --admin {shlex.quote(cfg.globus_admin)}"
if not cfg.collection_public: cmd += " --private"
stdout, stderr = run(node, cmd)

## Verify from outside

Reads Globus's own record of the endpoint, then checks from *this* machine - outside the VM -
that HTTPS answers on the public name and that SSH does **not** answer on the public address.

In [ ]:
import socket, urllib.request
slice, node, dev = ctx()
info = json.loads(node.execute("sudo cat /var/lib/globus-connect-server/info.json", quiet=True)[0])
domain = info["domain_name"]
node.execute(f"sudo bash -c '. {cfg.state_dir}/creds.env && export GCS_CLI_CLIENT_ID GCS_CLI_CLIENT_SECRET "
             f"GCS_CLI_ENDPOINT_ID=$(cat {cfg.state_dir}/endpoint-id.txt) && "
             f"globus-connect-server endpoint show && globus-connect-server collection list'")

def reachable(ip, port):
    """True = connected; False = refused/closed; None = this machine has no route."""
    fam = socket.AF_INET6 if ":" in ip else socket.AF_INET
    s = socket.socket(fam, socket.SOCK_STREAM); s.settimeout(5)
    try:
        s.connect((ip, port)); return True
    except ConnectionRefusedError:
        return False
    except OSError as e:
        # Unreachable network = this machine cannot test it (e.g. no IPv6 here).
        return None if getattr(e, "errno", None) in (51, 65, 101, 113) else False
    finally:
        s.close()

for ip in filter(None, (public_ip, public_ip6)):
    https, ssh = reachable(ip, 443), reachable(ip, 22)
    if https is None:
        print(f"{ip}: this machine has no route to it ({'IPv6' if ':' in ip else 'IPv4'}) - "
              f"run this check from a host that does, e.g. curl -6 https://{domain}/api/info")
        continue
    print(f"{ip}: 443 {'open (good)' if https else 'NOT reachable'}; "
          f"22 {'OPEN - the firewall step did not take' if ssh else 'closed (good)'}")
try:
    with urllib.request.urlopen(f"https://{domain}/api/info", timeout=15) as r:
        print(f"https://{domain}/api/info -> {r.status}")
except Exception as e:
    print(f"https://{domain}/api/info -> not from here ({e.__class__.__name__}); "
          f"an IPv6-only endpoint needs an IPv6-capable client")
coll = node.execute(f"sudo cat {cfg.state_dir}/collection-id.txt", quiet=True)[0].strip()
print(f"\nCollection: https://app.globus.org/file-manager?origin_id={coll}")

## Try a transfer

1. Open the collection link above and sign in with one of the identities in `users`.
2. You land in `data_base`; your writable directory is `<data_base>/<account>`.
3. Transfer a file in from another endpoint (or Globus Connect Personal), then check on the VM:
   `node.execute("sudo ls -l /data/<account>")`.

An identity that is not in `users` is refused at sign-in to the collection - that is the
identity map working. Without a Globus subscription, browser upload/download (HTTPS) and guest
collections are unavailable; endpoint-to-endpoint transfers work.

## Teardown

Delete the **endpoint before the slice**. Otherwise Globus keeps an endpoint pointing at an
address that no longer exists, and it can then only be removed by Globus support.

Two steps, in this order - `endpoint cleanup` refuses while a node is still attached:
`node cleanup` detaches this VM and stops its Globus services, then `endpoint cleanup` deletes
the endpoint (and its collections) from Globus. The deployment key is useless afterwards, so
the local copy is removed too.

In [ ]:
# slice, node, dev = ctx()
# pre = (f"sudo bash -c '. {cfg.state_dir}/creds.env && export GCS_CLI_CLIENT_ID GCS_CLI_CLIENT_SECRET && "
#        f"cd {cfg.state_dir} && ")
# run(node, pre + "globus-connect-server node cleanup'")
# run(node, pre + f"globus-connect-server endpoint cleanup --deployment-key {cfg.state_dir}/deployment-key.json "
#                 f"--agree-to-delete-endpoint'")
# os.remove(f"deployment-key-{cfg.site}.json")
# fablib.get_slice(name=cfg.slice_name).delete()